# 09: Release, policy certification and main results

Design decisions
- Frozen configurations, box tables (hashes from results/08_certificates.json) and candidate caches of notebook 08. A box is released only if it passed the second soundness check.
- Release rule (Eq. 8): among a patient's candidates whose bound meets the target of their class, the box with the fewest constrained features, then the most calibration records, then the tightest bound; otherwise abstain. Negative-outcome claims use alpha_0, enrichment claims alpha_1 = 1 - beta.
- Policy certification (Theorem 4) on the explained patients of calibration split 2 at the frozen targets: negative-outcome and enrichment releases are certified separately, each at delta_2 / 2 with the frozen alpha_sel. A kind whose policy is not certified is reported as "policy not certified".
- Test results on the 5,000 explained test patients: share of patients given a certified explanation, per kind; error rate among them with exact 95% intervals; explanation size and calibration support. Other values of alpha_0 and beta on the frozen grids are reported as descriptive rows; only the frozen targets carry the policy guarantee.
- Consistency check on the full test split: for every box released to a test patient, the test records inside it and their error rate. An estimate above the target is flagged when its exact lower limit, corrected for the number of released boxes, exceeds the target; this is a consistency check, not proof that a certificate is false.
- Underlying model on the full test split: AUROC, AUPRC, Brier score, ECE (15 equal-mass bins), calibration slope and intercept, sensitivity and specificity at tau.

In [ ]:
# === SESSION BOOTSTRAP (run first) ===
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'z3-solver'], check=True)
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
DRIVE_ROOT = '/content/drive/MyDrive/SCAX_Research'
REPO_DIR = f'{DRIVE_ROOT}/scax-research'
for name in ('.gitconfig', '.git-credentials'):
    if os.path.exists(f'{DRIVE_ROOT}/{name}'):
        shutil.copy(f'{DRIVE_ROOT}/{name}', os.path.expanduser(f'~/{name}'))
os.chdir(REPO_DIR)
os.environ['SCAX_DRIVE_ROOT'] = DRIVE_ROOT
sys.path.insert(0, f'{REPO_DIR}/src')
pull = subprocess.run(['git', 'pull', 'origin', 'main'], capture_output=True, text=True)
print(pull.stdout.strip() or pull.stderr.strip())

In [ ]:
import json, time
import numpy as np
import pandas as pd
from scax import paths
from scax.utils import seed_everything, sha256_file, write_json
from scax.data import public, splits
from scax.experiments.main_results import FrozenTask
from scax.explain.candidates import CandidateGenerator

SEED = 42
seed_everything(SEED)
TASKS = ['cdc_diabetes', 'brfss2015_heart']
REC05 = json.load(open(f'{paths.RESULTS}/05_frozen.json'))
assert REC05['written_before_calibration_split_1']
FT = {t: FrozenTask(t, REPO_DIR, DRIVE_ROOT, REC05['frozen_config_sha256'][t]) for t in TASKS}
CERT_DIR = paths.DERIVED / 'certificates'; CERT_DIR.mkdir(parents=True, exist_ok=True)


def load_task(task):
    X, y = public.LOADERS[task](paths.RAW)
    code = np.load(paths.DERIVED / 'splits' / f'{task}.npy')
    assert splits.summary(code)['sha256'] == FT[task].cfg['split_sha256'], 'split differs from the frozen one'
    return X.to_numpy(dtype=float), y, code


def generator(ft):
    c = ft.cfg['candidates']
    gen = CandidateGenerator(ft.pred, M=c['M'], seed=c['order_seed'], node_budget=c['node_budget'])
    assert gen.settings['inflation'] == c['inflation']
    return gen


for t in TASKS:
    print(t, 'frozen configuration', FT[t].cfg_sha[:16], '| model', FT[t].cfg['model']['sha256'][:16],
          '| prior', FT[t].cfg['prior']['sha256'][:16], '| alpha_sel', FT[t].targets['alpha_sel'])

from scax.certify.counting import box_counts
from scax.certify.clopper_pearson import cp_lower, cp_upper
from scax.experiments.main_results import pair_arrays, release_patients, kind_summary, policy_test
from scax.metrics.model_metrics import model_metrics
REC08 = json.load(open(f'{paths.RESULTS}/08_certificates.json'))

## Box tables and candidate sets from notebook 08

In [ ]:
STATE = {}
for task in TASKS:
    ft = FT[task]
    f = CERT_DIR / f'{task}__boxes.parquet'
    assert sha256_file(f) == REC08['box_table_sha256'][task], 'box table differs from notebook 08'
    df = pd.read_parquet(f)
    gen = generator(ft)
    assert gen.load(CERT_DIR / f'{task}__candidates.pkl')
    keys = [tuple(tuple(t) for t in json.loads(k)) for k in df.key]
    box_index = {(k, int(c)): i for i, (k, c) in enumerate(zip(keys, df.box_class))}
    U = np.where(df.verified.to_numpy() == 'sound', df.U.to_numpy(), 1.0)      # unverified boxes are never released
    Xa, y, code = load_task(task)
    parts = {}
    for name, rows in (('cal2', ft.cal2_rows), ('test', ft.test_rows)):
        C = ft.grid.cells(Xa[rows]); yhat = ft.pred.predict_cells(C).astype(int)
        lists = [gen.cache[gen.cell_key(c)] for c in C]
        pp, pb = pair_arrays(lists, box_index, ft.M)
        parts[name] = {'rows': rows, 'C': C, 'yhat': yhat, 'y': y[rows], 'pp': pp, 'pb': pb}
    STATE[task] = {'df': df, 'keys': keys, 'U': U, 'parts': parts, 'Xa': Xa, 'y': y, 'code': code}
    print(task, {n: len(p['rows']) for n, p in parts.items()}, 'boxes', len(df), 'verified sound', int((df.verified == 'sound').sum()))

## Policy certification on calibration split 2 and results on the test split

In [ ]:
MAIN, POLICY, RELEASED = [], {}, {}
for task in TASKS:
    ft, st = FT[task], STATE[task]
    df, U = st['df'], st['U']
    size, cls, n_B = df['size'].to_numpy(), df.box_class.to_numpy(), df.n_B.to_numpy()
    T = ft.targets
    settings = [(a0, b) for a0 in T['alpha_0_grid'] for b in T['beta_grid']]
    for a0, beta in settings:
        a1 = round(1 - beta, 10)
        is_main = (a0 == T['alpha_0']) and (beta == T['beta'])
        out = {}
        for name in ('cal2', 'test'):
            p = st['parts'][name]
            out[name] = release_patients(p['pp'], p['pb'], len(p['rows']), size, cls, n_B, U, a0, a1)
        pol = policy_test(out['cal2'], st['parts']['cal2']['yhat'], st['parts']['cal2']['y'],
                          {0: T['alpha_sel']['negative_outcome'], 1: T['alpha_sel']['enrichment']}, T['delta_2'])
        if is_main:
            POLICY[task] = pol
            RELEASED[task] = out['test']
        pt = st['parts']['test']
        ks = kind_summary(out['test'], pt['yhat'], pt['y'])
        for k, kind in ((0, 'negative outcome'), (1, 'enrichment')):
            rel = out['test']; a = (rel >= 0) & (pt['yhat'] == k); rb = rel[a]
            MAIN.append({'task': task, 'kind': kind, 'alpha_0': a0, 'beta': beta, 'target': a0 if k == 0 else a1,
                         'frozen_targets': is_main, 'cal2_accepted': pol[k]['accepted'], 'cal2_errors': pol[k]['errors'],
                         'U_sel': pol[k]['U_sel'], 'alpha_sel': pol[k]['alpha_sel'],
                         'policy_certified': pol[k]['certified'] if is_main else None,
                         'test_patients': len(pt['rows']), 'test_certified_share': ks[k]['share_of_patients'],
                         'test_certified_share_of_predicted_class': float(a.sum() / max(1, (pt['yhat'] == k).sum())),
                         'test_accepted': ks[k]['accepted'], 'test_errors': ks[k]['errors'], 'test_error_rate': ks[k]['error_rate'],
                         'test_error_lower_95': ks[k]['error_lower_95'], 'test_error_upper_95': ks[k]['error_upper_95'],
                         'mean_size': float(size[rb].mean()) if a.any() else None,
                         'median_calibration_support': float(np.median(n_B[rb])) if a.any() else None})
MAIN = pd.DataFrame(MAIN)
MAIN.to_csv(f'{paths.RESULTS}/09_main.csv', index=False)
print(MAIN[MAIN.frozen_targets].round(4).T.to_string())
for task in TASKS:
    for k, kind in ((0, 'negative outcome'), (1, 'enrichment')):
        p = POLICY[task][k]
        print(f"{task}, {kind}: U_sel {p['U_sel']:.4f} against alpha_sel {p['alpha_sel']} on {p['accepted']} accepted "
              f"-> {'certified' if p['certified'] else 'policy not certified'}")

## Released explanations on the test split

In [ ]:
REL_ROWS = []
for task in TASKS:
    ft, st = FT[task], STATE[task]; df = st['df']; pt = st['parts']['test']; rel = RELEASED[task]
    for i in range(len(rel)):
        b = int(rel[i])
        REL_ROWS.append({'task': task, 'row': int(pt['rows'][i]), 'predicted': int(pt['yhat'][i]), 'released': b >= 0,
                         'explanation': ft.grid.describe(ft.grid.from_key(st['keys'][b])) if b >= 0 else None,
                         'size': int(df['size'].iat[b]) if b >= 0 else None, 'n_B': int(df.n_B.iat[b]) if b >= 0 else None,
                         'U': float(df.U.iat[b]) if b >= 0 else None})
REL = pd.DataFrame(REL_ROWS)
REL.to_csv(f'{paths.RESULTS}/09_test_releases.csv', index=False)
for task in TASKS:
    for k, kind in ((0, 'negative outcome'), (1, 'enrichment')):
        r = REL[(REL.task == task) & REL.released & (REL.predicted == k)]
        print(f'\n{task}, {kind}: most frequent released explanations')
        print(r.explanation.value_counts().head(8).to_string())

## Consistency check inside released boxes on the full test split

In [ ]:
CONS = []
for task in TASKS:
    ft, st = FT[task], STATE[task]; df = st['df']; rel = RELEASED[task]
    test_all = splits.indices(st['code'], 'test')
    Ct, yt = ft.grid.cells(st['Xa'][test_all]), st['y'][test_all]
    ids = np.unique(rel[rel >= 0])
    n_t, k_t = box_counts([st['keys'][i] for i in ids], df.box_class.to_numpy()[ids], Ct, yt)
    K = max(1, len(ids))
    a0, a1 = ft.targets['alpha_0'], ft.targets['alpha_1']
    for j, i in enumerate(ids):
        target = a1 if df.box_class.iat[i] == 1 else a0
        low = float(cp_lower(int(k_t[j]), int(n_t[j]), 0.05 / K)) if n_t[j] else 0.0
        CONS.append({'task': task, 'box': df.key.iat[i], 'box_class': int(df.box_class.iat[i]), 'size': int(df['size'].iat[i]),
                     'calibration_n': int(df.n_B.iat[i]), 'calibration_k': int(df.k_B.iat[i]), 'U': float(df.U.iat[i]),
                     'target': target, 'test_n': int(n_t[j]), 'test_k': int(k_t[j]),
                     'test_error': float(k_t[j] / n_t[j]) if n_t[j] else None,
                     'test_upper_95': float(cp_upper(int(k_t[j]), int(n_t[j]), 0.05)) if n_t[j] else 1.0,
                     'test_lower_corrected': low, 'estimate_above_target': bool(low > target),
                     'test_share_inside': float(n_t[j] / len(test_all))})
CONS = pd.DataFrame(CONS)
CONS.to_csv(f'{paths.RESULTS}/09_box_consistency.csv', index=False)
print(CONS.groupby(['task', 'box_class']).agg(released_boxes=('box', 'size'), median_test_n=('test_n', 'median'),
                                              mean_test_error=('test_error', 'mean'), flagged=('estimate_above_target', 'sum')).to_string())

## Underlying model on the full test split

In [ ]:
METRICS = []
for task in TASKS:
    ft, st = FT[task], STATE[task]
    test_all = splits.indices(st['code'], 'test')
    margin = ft.pred.score(st['Xa'][test_all])
    p = 1 / (1 + np.exp(-margin))
    METRICS.append({'task': task, **model_metrics(p, st['y'][test_all], ft.pred.predict(st['Xa'][test_all]))})
METRICS = pd.DataFrame(METRICS)
METRICS.to_csv(f'{paths.RESULTS}/09_model_metrics.csv', index=False)
print(METRICS.round(4).T.to_string())

## Figure: certified share against the target

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 9, 'axes.spines.top': False, 'axes.spines.right': False})
NAMES = {'cdc_diabetes': 'CDC Diabetes', 'brfss2015_heart': 'BRFSS Heart disease'}
COLORS = {'cdc_diabetes': '#0072B2', 'brfss2015_heart': '#D55E00'}
fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.9))
for task in TASKS:
    T = FT[task].targets
    neg = MAIN[(MAIN.task == task) & (MAIN.kind == 'negative outcome') & (MAIN.beta == T['beta'])].sort_values('alpha_0')
    enr = MAIN[(MAIN.task == task) & (MAIN.kind == 'enrichment') & (MAIN.alpha_0 == T['alpha_0'])].sort_values('beta')
    axes[0].plot(neg.alpha_0, neg.test_certified_share, marker='o', lw=2, color=COLORS[task], label=NAMES[task])
    axes[1].plot(enr.beta, enr.test_certified_share, marker='o', lw=2, color=COLORS[task], label=NAMES[task])
axes[0].set_xlabel('Target error for negative-outcome claims'); axes[0].set_ylabel('Share of test patients')
axes[0].set_title('Certified negative-outcome explanations', fontsize=9)
axes[1].set_xlabel('Claimed minimum share with the outcome'); axes[1].set_title('Certified enrichment explanations', fontsize=9)
for ax in axes:
    ax.set_ylim(0, 1)
axes[0].legend(frameon=False, loc='upper left')
fig.tight_layout()
for ext in ('png', 'pdf'):
    fig.savefig(paths.FIGURES / f'fig_certified_share_vs_target.{ext}', dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
write_json(f'{paths.RESULTS}/09_policy_certification.json', {
    'frozen_config_sha256': {t: FT[t].cfg_sha for t in TASKS}, 'box_table_sha256': REC08['box_table_sha256'],
    'policy': {t: {('negative_outcome' if k == 0 else 'enrichment'): v for k, v in POLICY[t].items()} for t in TASKS},
    'main': MAIN[MAIN.frozen_targets].to_dict('records'), 'model_metrics': METRICS.to_dict('records'),
    'consistency_flags': {t: int(CONS[CONS.task == t].estimate_above_target.sum()) for t in TASKS}})
print('written: results/09_main.csv, 09_policy_certification.json, 09_test_releases.csv, 09_box_consistency.csv, 09_model_metrics.csv')

In [ ]:
# === STANDARD COMMIT CELL ===
import os, subprocess
os.chdir(REPO_DIR)
r = subprocess.run(["python", "tools/commit_cell.py",
  "09: release, policy certification on calibration split 2 and test results for tasks B and A, consistency check, model metrics, figure"],
  capture_output=True, text=True)
print(r.stdout); print(r.stderr)